# E-commerce Recommendation System

## Data Science Capstone Project

**Dataset:** E-commerce Behavior Data from Multi-Category Store

**Analysis Period:** October 2019

**Project Date:** August 2026

**Author:** Said Uzun

## Objective

This notebook develops a recommendation system using the user-product interaction dataset created in the previous stage of the project.

The main objectives are:

- Prepare the interaction data for recommendation modeling.
- Build a recommendation model based on implicit user feedback.
- Generate personalized product recommendations.
- Demonstrate example recommendations for selected users.

## Import Libraries

The required Python libraries are imported for data preparation and recommendation modeling.

In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

> **Note:** This notebook requires the `user_product_interactions.csv` file generated in Notebook 2.  
> If the Colab runtime is restarted, the file must be uploaded again before running the recommendation pipeline.

In [2]:
interactions = pd.read_csv("user_product_interactions.csv")

interactions.head()

,user_id,product_id,views,carts,purchases,interaction_score
0,541312140,44600062,2.0,0.0,0.0,2.0
1,554748717,3900821,1.0,0.0,0.0,1.0
2,550978835,31500053,1.0,0.0,0.0,1.0
3,555158050,2900536,1.0,0.0,0.0,1.0
4,530282093,1005011,1.0,0.0,0.0,1.0


In [3]:
print("Rows:", len(interactions))
print("Unique users:", interactions["user_id"].nunique())
print("Unique products:", interactions["product_id"].nunique())

Rows: 23307630
Unique users: 3022290
Unique products: 166794


In [4]:
interactions.describe()

,user_id,product_id,views,carts,purchases,interaction_score
count,2.330763e+07,2.330763e+07,2.330763e+07,2.330763e+07,2.330763e+07,2.330763e+07
mean,5.339573e+08,1.223632e+07,1.749616e+00,3.975162e-02,3.187149e-02,2.028228e+00
std,1.860199e+07,1.281032e+07,1.933272e+00,4.334675e-01,2.945564e-01,3.639464e+00
min,3.386938e+07,1.000978e+06,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
25%,5.160907e+08,1.306747e+06,1.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
50%,5.306347e+08,6.000163e+06,1.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
75%,5.520869e+08,2.140144e+07,2.000000e+00,0.000000e+00,0.000000e+00,2.000000e+00
max,5.662809e+08,6.050001e+07,3.110000e+02,2.360000e+02,2.480000e+02,2.199000e+03


In [5]:
interactions["interaction_score"].describe()

count    2.330763e+07
mean     2.028228e+00
std      3.639464e+00
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      2.000000e+00
max      2.199000e+03
Name: interaction_score, dtype: float64

The interaction dataset contains over 23 million user-product relationships generated from implicit feedback. Each interaction is represented by a weighted interaction score based on user behavior.

## Building the User–Product Interaction Matrix

Recommendation algorithms require the interaction data to be represented as a matrix where each row corresponds to a user, each column corresponds to a product, and each cell contains the interaction score between them.

Since the dataset is extremely sparse, a sparse matrix representation is used to reduce memory usage and improve computational efficiency.

In [6]:
user_codes = interactions["user_id"].astype("category")
product_codes = interactions["product_id"].astype("category")

In [7]:
interaction_matrix = csr_matrix(
    (
        interactions["interaction_score"],
        (
            user_codes.cat.codes,
            product_codes.cat.codes
        )
    )
)

interaction_matrix

<3022290x166794 sparse matrix of type '<class 'numpy.float64'>'
	with 23307630 stored elements in Compressed Sparse Row format>

In [8]:
print("Matrix shape:", interaction_matrix.shape)
print("Non-zero interactions:", interaction_matrix.nnz)

Matrix shape: (3022290, 166794)
Non-zero interactions: 23307630


## Normalizing User Vectors

Before computing similarities, each user's interaction vector is normalized.

This prevents highly active users from dominating similarity calculations simply because they interacted with many products. The recommendation model will focus on behavioral patterns rather than interaction volume.

In [10]:
from sklearn.preprocessing import normalize

normalized_matrix = normalize(interaction_matrix)

In [11]:
normalized_matrix

<3022290x166794 sparse matrix of type '<class 'numpy.float64'>'
	with 23307630 stored elements in Compressed Sparse Row format>

Each row now has unit length, allowing cosine similarity to compare users based on interaction patterns instead of absolute interaction counts.

## Item-Based Collaborative Filtering

To make the recommendation process computationally efficient, the model uses item-based collaborative filtering.

Products are considered similar when they receive similar interaction patterns from users. Cosine similarity is used to measure the similarity between product interaction vectors.

In [12]:
product_support = (
    interactions
    .groupby("product_id")["user_id"]
    .nunique()
    .sort_values(ascending=False)
)

top_products = product_support.head(5000).index

model_interactions = interactions[
    interactions["product_id"].isin(top_products)
].copy()

print("Model rows:", len(model_interactions))
print("Model users:", model_interactions["user_id"].nunique())
print("Model products:", model_interactions["product_id"].nunique())

Model rows: 14471841
Model users: 2692324
Model products: 5000


## Training the Recommendation Model

A K-Nearest Neighbors (KNN) model with cosine distance is used to identify similar products based on user interaction patterns.

The model does not require explicit ratings and therefore is well suited for implicit feedback recommendation systems.

In [13]:
item_codes = model_interactions["product_id"].astype("category")
user_codes = model_interactions["user_id"].astype("category")

item_user_matrix = csr_matrix(
    (
        model_interactions["interaction_score"],
        (
            item_codes.cat.codes,
            user_codes.cat.codes
        )
    )
)

item_user_matrix

<5000x2692324 sparse matrix of type '<class 'numpy.float64'>'
	with 14471841 stored elements in Compressed Sparse Row format>

In [14]:
print("Item matrix shape:", item_user_matrix.shape)

Item matrix shape: (5000, 2692324)


In [15]:
knn_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=6
)

knn_model.fit(item_user_matrix)

NearestNeighbors(algorithm='brute', metric='cosine', n_neighbors=6)

## Similar Product Recommendations

The trained model is used to identify products with similar user interaction patterns.  
Cosine distance is converted into a similarity score, where values closer to 1 indicate stronger similarity.

In [16]:
product_ids = item_codes.cat.categories.to_numpy()

product_to_index = pd.Series(
    np.arange(len(product_ids)),
    index=product_ids
)

In [17]:
def get_similar_products(product_id, n=5):

    if product_id not in product_to_index.index:
        return pd.DataFrame()

    product_index = int(product_to_index.loc[product_id])

    distances, indices = knn_model.kneighbors(
        item_user_matrix[product_index],
        n_neighbors=n + 1
    )

    similar_indices = indices.flatten()[1:]
    similarities = 1 - distances.flatten()[1:]

    return pd.DataFrame({
        "product_id": product_ids[similar_indices],
        "similarity_score": similarities
    })

In [18]:
example_product = top_products[0]

print("Target Product:", example_product)

similar_products = get_similar_products(
    example_product,
    n=5
)

similar_products

Target Product: 1004856


,product_id,similarity_score
0,1004833,0.255787
1,1004767,0.175866
2,1004857,0.163422
3,1005100,0.160857
4,1004858,0.154653


## Personalized Product Recommendations

Personalized recommendations are generated by combining the products a user has previously interacted with and the most similar products identified by the item-based collaborative filtering model.

Products already seen by the user are excluded from the final recommendation list.

In [19]:
def recommend_for_user(user_id, n=5, neighbors_per_item=10):

    user_history = model_interactions[
        model_interactions["user_id"] == user_id
    ]

    if user_history.empty:
        return pd.DataFrame()

    seen_products = set(user_history["product_id"])

    recommendation_scores = {}

    for _, row in user_history.iterrows():

        product_id = row["product_id"]
        interaction_score = row["interaction_score"]

        if product_id not in product_to_index.index:
            continue

        product_index = int(product_to_index.loc[product_id])

        distances, indices = knn_model.kneighbors(
            item_user_matrix[product_index],
            n_neighbors=neighbors_per_item + 1
        )

        for distance, neighbor_index in zip(
            distances.flatten()[1:],
            indices.flatten()[1:]
        ):

            recommended_product = product_ids[neighbor_index]

            if recommended_product in seen_products:
                continue

            similarity = 1 - distance

            recommendation_scores[recommended_product] = (
                recommendation_scores.get(recommended_product, 0)
                + similarity * interaction_score
            )

    recommendations = pd.DataFrame(
        recommendation_scores.items(),
        columns=["product_id", "recommendation_score"]
    )

    return (
        recommendations
        .sort_values("recommendation_score", ascending=False)
        .head(n)
        .reset_index(drop=True)
    )

In [20]:
user_activity = (
    model_interactions
    .groupby("user_id")["product_id"]
    .nunique()
)

candidate_users = user_activity[
    (user_activity >= 10) &
    (user_activity <= 30)
]

example_user = candidate_users.index[0]

print("Example User:", example_user)
print("Products Interacted:", candidate_users.loc[example_user])

Example User: 239876607
Products Interacted: 12


In [21]:
user_recommendations = recommend_for_user(
    example_user,
    n=5
)

user_recommendations.round(3)

,product_id,recommendation_score
0,1005101,1.654
1,1005023,1.379
2,1005196,1.095
3,1004792,0.817
4,1005151,0.810


The recommendation model successfully generated personalized product suggestions for the selected user.

The recommended products were not previously interacted with by the user and were ranked according to the weighted similarity scores derived from item-based collaborative filtering.

## Conclusion

This notebook developed an item-based collaborative filtering recommendation system using implicit user feedback.

The recommendation pipeline included:

- building a sparse user-product interaction matrix,
- training a k-Nearest Neighbors model using cosine similarity,
- identifying similar products,
- generating personalized product recommendations based on weighted similarity scores.

The resulting recommendation engine demonstrates how behavioral interaction data can be transformed into meaningful product suggestions without requiring explicit user ratings.